# v65 RealMLP on CPU -- a different FUNCTION CLASS on v58's features

**Why.** NOTES 3.107 measured which changes actually decorrelate. Architecture is the only
lever that reaches below corr 0.94; an algorithm swap (CatBoost vs XGBoost on identical
features) only reached 0.9882 and is dead. RealMLP is a genuinely different function class
-- smooth and global, where trees are piecewise-constant.

**The bar.** A new member helps v62 (0.665506) iff `corr(e_i,e_j) < s_i/s_j`. RealMLP keeps
v58's engineered features rather than consuming a raw sequence, so it should decorrelate
*less* than the GRU's 0.7865 -- plausibly **0.94-0.97**, meaning it must score under
**0.686-0.700** to pay. Achievable for RealMLP on tabular data, but a narrower margin than
the sequence-model route.

**v69 = v68 at FRAC 0.20 (v58's own value). That is the only difference from v68.**
The 0.05 in v65/v68 was a compute compromise I introduced, and it confounded every
comparison against v58 with a 4x data difference. Early stopping made the full value
affordable: v68 ran in 43m33s at 0.05, so 0.20 should land near 2.5-3 h.

**v68 = v65 + EARLY STOPPING.** v65 scored 0.708551229 and its
own validation curve shows why: 0.6198 at epoch 139, **0.7270 at epoch 256, and epoch 256 is
what shipped**. `use_early_stopping` defaults off and I did not pass it. The 0.7086 was a
broken run, not a verdict on the model class -- I had attributed it to temporal
extrapolation and was wrong (NOTES 3.114, retracted).

**Held fixed on purpose:** v58's 132 features verbatim, the same delta-from-anchor target,
the same train/validation split, the same row-draw order, the same FRAC and VAL_CAP. Only
early stopping changes, so the comparison to v65 is clean.

⚠️ **Needs Internet ON** (notebook settings) for `pip install pytabkit`.
⚠️ **NaN handling is a real difference from XGBoost** -- see the `prep()` docstring.
⚠️ **Not determinism-guaranteed.** torch CPU with pinned seeds is far closer than any GPU
path, but is not bit-identical across thread counts the way XGBoost `hist` is. Report it
that way; do not claim bit-reproducibility for this model.


In [ ]:
from pathlib import Path
import os, json, time, subprocess, sys
import numpy as np, pandas as pd

# ⚠️ CPU ON PURPOSE -- set Accelerator to None. GPU is banned outright for this project
# (NOTES 3.104): the XGBoost GPU sketch is not order-pinned and reproducibility is 30% of
# the final score. torch on CPU with pinned seeds is far closer to deterministic than any
# GPU path, though not bit-identical -- see the honesty note at the bottom.
DATA_DIR = Path("/kaggle/input/datasets/lastopt/fgws25data/data_kaggle")
OUT_DIR  = Path("/kaggle/working")
STEM     = "sub_realmlp_v72"
TAG      = "v72_realmlp_e15"
NTHREAD  = 16          # kept at v58's value so the FEATURE code is bit-identical
SEED     = 42

# ✅ FRAC 0.20 -- v58's OWN value, restored. The 0.05 in v65/v68 was a compute compromise
# I introduced, and it was a confound: v58 trains on 0.20, so any v65-vs-v58 comparison
# mixed "different learner" with "4x less data". Early stopping made 0.20 affordable --
# v68 finished in 43m33s at FRAC 0.05, so ~4x the rows lands near 2.5-3 h.
# ⚠️ NOTES memory records FRAC 0.20 as a SHARP optimum FOR XGBOOST (0.10 costs +0.005,
# 0.40 costs +0.006). Nothing says that transfers to an MLP; 0.20 is chosen here to match
# v58 and remove the confound, not because it is known to be optimal for this model.
# ⚠️⚠️ THAT ESTIMATE IS ARITHMETIC, NOT A MEASUREMENT, and my last runtime estimate for
# this exact hardware was wrong by 6x (I predicted 8-9 h for a CPU XGBoost run that took
# 87 min). `fit()` is one blocking call, so THERE IS NO WALL-CLOCK GUARD -- if it overruns,
# the Kaggle session dies with no submission. Mitigation: verbosity=2 prints per-epoch
# progress, so check the first few epochs and kill it early if the extrapolation exceeds
# your session limit. EPOCHS_OVERRIDE below is the dial to turn if it does.
FRAC_OVERRIDE   = 0.20
VAL_CAP         = 200_000   # pytabkit folds X_val into the fit; the raw draw is 1.11M rows
EPOCHS_OVERRIDE = 15      # the real kwarg is `n_epochs`; None keeps RealMLP's tuned
                            # default, which is the whole point of it -- prefer lowering FRAC
BUDGET_S        = 8 * 3600  # passed to fit(time_to_fit_in_seconds=...). Kaggle CPU sessions
                            # cap at 12 h; this leaves room for feature building + predict.

PYTABKIT_PIN = "1.5.0"     # None accepts whatever pip resolves
try:
    import pytabkit  # noqa
except ImportError:
    spec = f"pytabkit=={PYTABKIT_PIN}" if PYTABKIT_PIN else "pytabkit"
    print(f"installing {spec} (needs Internet ON in the notebook settings) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
import pytabkit, torch
import importlib.metadata as _md

def _ver(mod, dist):
    """pytabkit ships no __version__; ask the package metadata instead."""
    v = getattr(mod, "__version__", None)
    if v: return v
    try:    return _md.version(dist)
    except Exception: return "unknown"

PYTABKIT_VER = _ver(pytabkit, "pytabkit")
print(f"pytabkit {PYTABKIT_VER}  torch {torch.__version__}  "
      f"cuda_available={torch.cuda.is_available()}  cores={os.cpu_count()}")
if PYTABKIT_VER not in (PYTABKIT_PIN, "unknown"):
    print(f"  ⚠️ pin is {PYTABKIT_PIN} but {PYTABKIT_VER} is installed -- "
          "results are not comparable to a run at the pinned version.")

torch.manual_seed(SEED); np.random.seed(SEED)
torch.set_num_threads(min(os.cpu_count() or 4, 4))

if not (DATA_DIR / "raw/Train.csv").exists():
    seen = sorted(str(q) for q in Path("/kaggle/input").rglob("*") if q.is_dir())[:30]
    raise SystemExit(f"raw/Train.csv not under {DATA_DIR}. Present:\n" +
                     "\n".join("  " + s for s in seen))
import xgboost as xgb          # the feature module imports it; not used for training
print(f"{STEM}: FRAC={FRAC_OVERRIDE}  val_cap={VAL_CAP}  "
      f"epochs={EPOCHS_OVERRIDE or 'default'}  budget={BUDGET_S/3600:.1f} h")


In [ ]:
"""XGBoost head-to-head against the frozen LightGBM v5+seeds5.

Same 73 v5 features, same delta target, same test-mix horizon weights, same
train/validation split and the same 5-seed refit as sub_v5_seeds5.py. The ONLY
variable changed is the learner. This is a like-for-like head-to-head against
LightGBM, not a blend.

Baseline to beat: sub_v5_seeds5.py -- validation 0.6175 / best_iter 217, LB 0.693870116.

Capacity is matched to the LightGBM production config deliberately:
    LightGBM  num_leaves=255, min_data_in_leaf=200, lr .05, ff/bf 0.8, l2 1.0, max_bin 127
    XGBoost   grow_policy=lossguide + max_leaves=255 (leaf-wise, like LightGBM),
              colsample/subsample 0.8, lambda 1.0, max_bin 127.
              min_child_weight=30, NOT 200: it is a sum of HESSIANS, and the rows
              carry test-mix weights averaging ~1/7, so 200 rows ~ 29 hessian.

--- v58 -------------------------------------------------------------------
v47 (LB 0.673767576) with ONE term added to the water balance: RUNOFF. 132 features.

THE GAP. The mass balance of a water store is

    dS/dt = P - E - R

v36 won a slot (-0.0008) by fitting that relationship per cell, as-of, from ERA5 -- and it
is the ONLY ERA5 encoding that has ever won here. But it fits only `a*tp + b*e`. **The
runoff term is simply absent.** Measured over the whole panel:

    P (tp)        0.06713
    E (e)         0.04108
    |P| - |E|     0.02605   <- what v36/v47 model
    R (sro+ssro)  0.02606   <- what they OMIT      ratio 1.00x

The dropped term is the same size as the balance being modelled. ERA5 supplies `sro`
(surface) and `ssro` (subsurface) at 100% coverage; they are summed, because the balance
does not care which path the water left by.

WHY THIS IS NOT "ANOTHER ERA5 COLUMN", the failure mode that cost this project four slots.
v32/v33/v34/v44 all added ERA5 as raw levels or window sums and all lost (+0.0017 to
+0.0055; v44 is the worst regression of the project). The single win encoded ERA5 as
per-cell as-of REGRESSION COEFFICIENTS. v58 changes nothing about the encoding -- it
completes the equation that encoding is fitting. One new column, `e5_bal_c`; `e5_bal_a/b`,
`r2`, `bal_now` and `bal_win` all change value because the regression is better specified.

The solve goes 2x2 -> 3x3, by cofactors, still fully vectorised over rows. The scatter
matrix is symmetric so its adjugate is too and six cofactors suffice. Verified against
`numpy.linalg.solve` at max abs err 4.4e-16; a perfectly collinear runoff drives det to
exactly 0 and is caught by the existing 1e-12 guard.

WHAT WOULD FALSIFY IT. Runoff is the least observationally constrained field in this block:
`tp` is assimilated, `e` is semi-diagnostic, but `sro`/`ssro` are pure land-surface
parameterisation. This project's record is **observed beats modelled** -- CCI (retrieval)
and SPEI (index) won; modelled soil moisture lost twice (v50, and swvl inside v32). If v58
fails, the honest reading is that ERA5's modelled fluxes are exhausted at `tp`/`e`, not
that the mass balance is the wrong frame.

⚖️ THE 3x3 FALLS BACK TO v47's 2x2 where runoff is uninformative. Screened on 280,801 real
rows the bare 3x3 resolves for 84.0% against v47's 93.3%, because arid cells have ~zero
runoff and a singular scatter matrix. Shipping it bare would have STRIPPED `bal_a`/`bal_b`
from 9.3% of rows that already have them -- a regression disguised as a feature. With the
fallback, v58's block is a strict superset of v47's: three terms where runoff carries
information, v47's exact two where it does not (`bc` = 0 makes the formulas identical).

Runoff also correlates with precipitation, so the per-cell `a` coefficient may already
absorb part of R -- in which case the marginal gain is smaller than the 1.00x ratio
suggests. That is a reason to expect a modest effect, not no effect.

"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

# Paths resolve against the project root, so this runs from any directory.
ROOT = DATA_DIR

MONTH0 = 2002 * 12 + 4
N_MONTHS = (2019 * 12 + 0) - MONTH0 + 1          # 201
COVARIATES = ["SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t", "SOIL_MOISTURE_t"]
SHORT = ["spei01", "spei03", "spei06", "spei12", "sm"]
TWS_LAGS = (1, 3, 6, 12)
ROLL = 12
MAX_H = 7
RESP_COV = [1, 3, 4]      # SPEI_03, SPEI_12, SOIL_MOISTURE
CURVE_MIN_N = 24           # months of own TWS needed before a quadratic is trusted
CURVE_U0, CURVE_US = 100.0, 100.0   # time is rescaled to u=(m-100)/100 in [-1,1]:
                           # a raw-month Vandermonde (m^4 up to 1.6e9) is hopeless.
ECM_COV = [3, 4]           # SPEI_12, SOIL_MOISTURE -- the two STOCK-like state
                           # variables. SPEI_03 is a flow and v19 already covers it.
NBR_COV = [2, 3, 4]        # SPEI_06, SPEI_12, SOIL_MOISTURE at rings 1/2/3
R1_EXTRA = [0, 1]          # SPEI_01, SPEI_03 at ring 1 only
E5_PATH = "era5_grid.parquet"      # regridded ERA5-Land, 15,715 x 204
CCI_PATH = "cci_grid.parquet"      # regridded ESA CCI satellite SM
DR_PATH  = "drought_grid.parquet"  # regridded CDS SPEI 12/24/36/48
E5_WIN = ["tp", "e", "ro"]              # accumulated over (anchor, t]
                                        # "ro" is DERIVED: sro + ssro
VAL_MONTHS, FRAC, VAL_FRAC, SEED, ROUNDS = 24, 0.20, 0.50, 42, 3000
PARAMS = dict(objective="reg:squarederror", eval_metric="rmse", learning_rate=0.05,
              grow_policy="lossguide", max_leaves=255, max_depth=0,
              min_child_weight=30.0, colsample_bytree=0.8, subsample=0.8,
              reg_lambda=1.0, tree_method="hist", max_bin=127, nthread=NTHREAD,
              seed=SEED)
N_SEEDS = 5

NAMES = (
    ["tws_anchor", "h", "clim_anchor", "clim_target", "anom_anchor", "clim_delta",
     "cell_mean", "cell_std", "z_anchor", "cell_trend", "trend_proj",
     "tws_roll_mean_12", "anchor_minus_roll12"]
    + [f"d_tws_{k}" for k in TWS_LAGS] + [f"gap_tws_{k}" for k in TWS_LAGS]
    + [f"{c}_t" for c in SHORT] + [f"{c}_anchor" for c in SHORT]
    + [f"d_{c}" for c in SHORT] + [f"{c}_win_mean" for c in SHORT]
    + ["month_sin_target", "month_cos_target"]
    + ["nbr_tws_anchor", "nbr_minus_own", "nbr_d_tws_1", "nbr_d_tws_3"]
    + [f"nbr_{SHORT[i]}_t" for i in NBR_COV] + [f"nbr_d_{SHORT[i]}" for i in NBR_COV]
    + [f"nbr_{SHORT[i]}_t" for i in R1_EXTRA] + [f"nbr_d_{SHORT[i]}" for i in R1_EXTRA]
    + [f"nbr2_{SHORT[i]}_t" for i in NBR_COV] + [f"nbr2_d_{SHORT[i]}" for i in NBR_COV]
    + ["grad_spei06", "grad_sm"]
    + [f"nbr3_{SHORT[i]}_t" for i in NBR_COV] + [f"nbr3_d_{SHORT[i]}" for i in NBR_COV]
    + ["grad2_spei06", "grad2_sm"]
    # --- v16: volatility of the cell's OWN past TWS. Guidelines section 3 permits
    #     "per-cell statistics computed from that cell's own past TWS: climatology,
    #     variance, trend" explicitly. Motivation (NOTES 3.29): the model predicts
    #     moves of rms 0.49 in BOTH calm and volatile regimes while actual moves are
    #     0.71 and 1.03 -- it cannot tell the regimes apart, so it cannot shrink
    #     harder where uncertainty is genuinely higher.
    + ["dvol_all", "dvol_12", "dvol_ratio", "dvol_h", "clim_std_target",
       "z_clim_anchor", "d1_over_vol"]
    + [f"{k}_{n}" for n in ("spei03", "spei12", "sm") for k in ("rho", "beta", "imp")]
    + ["rho_ar", "beta_ar", "imp_ar", "rho_mr", "beta_mr", "imp_mr", "mr_decay_h"]
    + [f"ecm_{k}_{n}" for n in ("spei12", "sm")
       for k in ("k", "g", "gap", "dec")]
    + ["curve_c", "slope_now", "slope_now_proj", "curve_move", "curve_vs_lin"]
    + ["dvol_tm", "dvol_tm_ratio", "seas_amp", "seas_pos_t", "seas_rising_t",
       "anom_vs_tm", "anom_vs_tm_over_dvol"]
    + ["amr_x_n3spei06", "z_x_n3spei06", "tws_x_n3spei06", "anomtm_x_n3spei06",
       "amr_x_z"]
    # --- v36 ERA5 joint per-cell water balance ---
    + ["e5_bal_a", "e5_bal_b", "e5_bal_c", "e5_bal_r2", "e5_bal_now", "e5_bal_win"]
    # --- v45 ESA CCI satellite soil moisture ---
    + ["cci_vs_clim", "d_cci"]
    # --- v47 long-window SPEI ---
    + ["d_spei24", "d_spei48", "spei12_m_48"]
)


def log(m):
    import time
    print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)


class P:
    """(cell x month) panels and every as-of-month statistic built on them."""

    def __init__(self, tr, te):
        gid = lambda la, lo: (np.rint(la + 55.5).astype(np.int64) * 360
                              + np.rint(lo + 179.5).astype(np.int64))
        g_tr = gid(tr["lat"].to_numpy(), tr["lon"].to_numpy())
        g_te = gid(te["lat"].to_numpy(), te["lon"].to_numpy())
        codes, uniq = pd.factorize(np.concatenate([g_tr, g_te]), sort=True)
        self.n = len(uniq); self.gids = np.asarray(uniq, dtype=np.int64)
        self.c_tr = codes[:len(g_tr)].astype(np.int32)
        self.c_te = codes[len(g_tr):].astype(np.int32)
        mid = lambda s: (s.astype("string").str.slice(0, 4).astype("int32").to_numpy() * 12
                         + s.astype("string").str.slice(5, 7).astype("int32").to_numpy()
                         - 1 - MONTH0).astype(np.int32)
        self.m_tr, self.m_te = mid(tr["time"]), mid(te["time"])

        # --- TWS from its three consistent sources -------------------------
        t = np.full((self.n, N_MONTHS), np.nan, np.float32)
        t[self.c_tr, self.m_tr] = tr["TWS_t"].to_numpy()
        nx = self.m_tr + 1; ok = nx < N_MONTHS
        t[self.c_tr[ok], nx[ok]] = tr["target"].to_numpy()[ok]
        v = te["TWS_t"].to_numpy(); s = ~np.isnan(v)
        t[self.c_te[s], self.m_te[s]] = v[s]
        self.tws = t

        self.cov = []
        for c in COVARIATES:
            m = np.full((self.n, N_MONTHS), np.nan, np.float32)
            m[self.c_tr, self.m_tr] = tr[c].to_numpy()
            m[self.c_te, self.m_te] = te[c].to_numpy()
            self.cov.append(m)

        obs = ~np.isnan(t)
        idx = np.where(obs, np.arange(N_MONTHS, dtype=np.int32)[None, :], np.int32(-1))
        self.last = np.maximum.accumulate(idx, axis=1).astype(np.int32)

        y = np.where(obs, t, 0.0).astype(np.float64)
        x = np.arange(N_MONTHS, dtype=np.float64)[None, :] * obs
        n = np.cumsum(obs, axis=1, dtype=np.float64)
        sy, syy = np.cumsum(y, 1), np.cumsum(y * y, 1)
        sx, sxx, sxy = np.cumsum(x, 1), np.cumsum(x * x, 1), np.cumsum(x * y, 1)
        self.csum, self.ccnt = sy.astype(np.float32), n.astype(np.float32)
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = sy / n; var = syy / n - mean * mean
            den = n * sxx - sx * sx
            slope = np.where(np.abs(den) > 1e-9, (n * sxy - sx * sy) / den, np.nan)
        bad = n < 2
        self.cmean = np.where(n < 1, np.nan, mean).astype(np.float32)
        self.cstd = np.where(bad, np.nan, np.sqrt(np.maximum(var, 0))).astype(np.float32)
        self.ctrend = np.where(bad, np.nan, slope).astype(np.float32)

        # month-to-month change volatility, from this cell's own past TWS only
        prev = np.full_like(t, np.nan); prev[:, 1:] = t[:, :-1]
        dif = t - prev                     # NaN unless both months observed
        od = ~np.isnan(dif)
        sdd = np.cumsum(np.where(od, dif * dif, 0.0), 1, dtype=np.float64)
        ndd = np.cumsum(od, 1, dtype=np.float64)
        with np.errstate(invalid="ignore", divide="ignore"):
            self.dvol = np.where(ndd >= 3, np.sqrt(sdd / ndd), np.nan).astype(np.float32)
        self.dsq, self.dcnt = sdd.astype(np.float32), ndd.astype(np.float32)

        # v19: per-cell RESPONSE coefficients. How strongly does this cell's storage
        # actually move when its soil moisture / drought index moves? Accumulated as-of,
        # over months where BOTH first differences are observed. `dif` is dTWS above.
        self.resp = {}
        for ci in RESP_COV:
            cprev = np.full_like(self.cov[ci], np.nan); cprev[:, 1:] = self.cov[ci][:, :-1]
            dc = self.cov[ci] - cprev
            ok = od & ~np.isnan(dc)
            cs_ = lambda a: np.cumsum(np.where(ok, a, 0.0), 1, dtype=np.float64).astype(np.float32)
            self.resp[ci] = (np.cumsum(ok, 1, dtype=np.float64).astype(np.float32),
                             cs_(dc), cs_(dif), cs_(dc * dc), cs_(dif * dif), cs_(dc * dif))

        # v20: per-cell DYNAMICS coefficients. `dif`/`od`/`prev` come from the v16 block.
        #   ar -- regress dTWS_t on dTWS_{t-1}: does this cell's change carry momentum
        #         or reverse?  mr -- regress dTWS_t on TWS_{t-1}: how hard does this cell
        #         pull back toward its own mean, and how fast?
        dprev = np.full_like(dif, np.nan); dprev[:, 1:] = dif[:, :-1]

        def _xmom(x, y):
            k = ~np.isnan(x) & ~np.isnan(y)
            c = lambda a: np.cumsum(np.where(k, a, 0.0), 1, dtype=np.float64).astype(np.float32)
            return (np.cumsum(k, 1, dtype=np.float64).astype(np.float32),
                    c(x), c(y), c(x * x), c(y * y), c(x * y))

        self.ar = _xmom(dprev, dif)
        self.mr = _xmom(prev, dif)

        # v22: per-cell ERROR CORRECTION. Regress dTWS_t on BOTH TWS_{t-1} and
        # Driver_{t-1}. v20's `mr` is the gamma=0 special case of this -- reversion
        # to a CONSTANT cell mean. Here the equilibrium is driver-conditional,
        # TWS* = -(gamma/k) * Driver, so a cell sitting in a wet anomaly is not
        # expected to fall back to its long-run mean at all. Differencing the LHS
        # keeps this out of spurious-regression territory: a levels-on-levels fit
        # would mostly recover co-trending.
        self.ecm = {}
        for ci in ECM_COV:
            dv_ = np.full_like(self.cov[ci], np.nan); dv_[:, 1:] = self.cov[ci][:, :-1]
            msk = ~np.isnan(prev) & ~np.isnan(dv_) & ~np.isnan(dif)
            cm_ = lambda a: np.cumsum(np.where(msk, a, 0.0), 1,
                                      dtype=np.float64).astype(np.float32)
            self.ecm[ci] = (np.cumsum(msk, 1, dtype=np.float64).astype(np.float32),
                            cm_(prev), cm_(dv_), cm_(dif),
                            cm_(prev * prev), cm_(prev * dv_), cm_(dv_ * dv_),
                            cm_(prev * dif), cm_(dv_ * dif))

        # v25: per-cell TREND CURVATURE. Every trend feature so far is LINEAR --
        # `cell_trend` is one slope over the cell's whole history and `trend_proj`
        # extrapolates it straight. But the test period is 18 months of pure forward
        # extrapolation (3.25) and real storage trends bend: aquifer depletion
        # accelerates, El Nino breaks the record. For a cell whose decline is
        # steepening, the LIFETIME-AVERAGE slope is the wrong number to extrapolate.
        # Fit TWS = a + b*u + c*u^2 as-of, over months where TWS is observed.
        u = ((np.arange(N_MONTHS, dtype=np.float64) - CURVE_U0) / CURVE_US)[None, :]
        uu = np.broadcast_to(u, t.shape)
        yq = np.where(obs, t, 0.0).astype(np.float64)
        qc = lambda a: np.cumsum(np.where(obs, a, 0.0), 1, dtype=np.float64)
        self.qd = (np.cumsum(obs, 1, dtype=np.float64),
                   qc(uu), qc(uu ** 2), qc(uu ** 3), qc(uu ** 4),
                   qc(yq), qc(uu * yq), qc(uu ** 2 * yq))

        self.covsum, self.covcnt = [], []
        for m in self.cov:
            o = ~np.isnan(m)
            self.covsum.append(np.cumsum(np.where(o, m, 0.0), 1, dtype=np.float64).astype(np.float32))
            self.covcnt.append(np.cumsum(o, 1, dtype=np.float64).astype(np.float32))

        cal = (np.arange(N_MONTHS) + MONTH0) % 12
        self.clim = np.empty((12, self.n, N_MONTHS), np.float32)
        for c in range(12):
            sel = cal == c
            a = np.cumsum(np.where(sel[None, :], y, 0.0), 1)
            k = np.cumsum(np.where(sel[None, :], obs, 0.0), 1)
            with np.errstate(invalid="ignore", divide="ignore"):
                self.clim[c] = np.where(k > 0, a / k, np.nan).astype(np.float32)

        # v26: per-CALENDAR-MONTH change volatility. Every per-cell parameter so far is
        # ONE number per cell, averaged over all seasons -- but a snow cell accumulates
        # in winter and dumps in melt season, and a monsoon cell fills and drains on a
        # schedule. One `dvol` averages those regimes together, the same way v19's single
        # beta averaged the wetting and drying branches.
        # NOTE `clim_std_target` (v16) is the spread of LEVELS in the target month; this
        # is the spread of CHANGES. That levels-vs-changes distinction is exactly what
        # v16 won on (3.33), applied one level deeper. Built as-of like climstd, so no
        # month at or after the target can enter -- a wet/dry mask fitted on the whole
        # record would encode months >= t+1 for training rows and is NOT used.
        self.dvolm = np.empty((12, self.n, N_MONTHS), np.float32)
        d2 = np.where(od, dif * dif, 0.0).astype(np.float64)
        for c in range(12):
            sel = (cal == c)[None, :] & od
            a = np.cumsum(np.where(sel, d2, 0.0), 1)
            k = np.cumsum(sel, 1)
            with np.errstate(invalid="ignore", divide="ignore"):
                self.dvolm[c] = np.where(k >= 2, np.sqrt(a / np.maximum(k, 1)),
                                         np.nan).astype(np.float32)

        self.climstd = np.empty((12, self.n, N_MONTHS), np.float32)
        for c in range(12):
            sel = cal == c
            a = np.cumsum(np.where(sel[None, :], y, 0.0), 1)
            q = np.cumsum(np.where(sel[None, :], y * y, 0.0), 1)
            k = np.cumsum(np.where(sel[None, :], obs, 0.0), 1)
            with np.errstate(invalid="ignore", divide="ignore"):
                mu = np.where(k > 0, a / k, np.nan)
                var = np.where(k > 1, q / k - mu * mu, np.nan)
            self.climstd[c] = np.sqrt(np.maximum(var, 0)).astype(np.float32)

        # --- neighbourhoods: lat/lon builds the adjacency table, nothing else
        lat_i, lon_i = self.gids // 360, self.gids % 360
        look = np.full(140 * 360, -1, np.int32); look[self.gids] = np.arange(self.n, dtype=np.int32)

        def ring(r):
            offs = [(a, b) for a in range(-r, r + 1) for b in range(-r, r + 1)
                    if max(abs(a), abs(b)) == r]
            out = np.full((self.n, len(offs)), -1, np.int32)
            for k, (dla, dlo) in enumerate(offs):
                la, lo = lat_i + dla, (lon_i + dlo) % 360
                ins = (la >= 0) & (la < 140)
                out[ins, k] = look[la[ins] * 360 + lo[ins]]
            return out

        def mean_over(panel, ix):
            vals = panel[np.where(ix >= 0, ix, 0)]; vals[ix < 0] = np.nan
            o = ~np.isnan(vals); cnt = o.sum(1); tot = np.where(o, vals, 0.0).sum(1)
            with np.errstate(invalid="ignore", divide="ignore"):
                return np.where(cnt > 0, tot / cnt, np.nan).astype(np.float32)

        r1, r2, r3 = ring(1), ring(2), ring(3)
        self.nbr_tws = mean_over(self.tws, r1)
        self.n1 = {k: mean_over(self.cov[k], r1) for k in [0, 1, 2, 3, 4]}
        self.n2 = {k: mean_over(self.cov[k], r2) for k in NBR_COV}
        self.n3 = {k: mean_over(self.cov[k], r3) for k in NBR_COV}

        # ---- v36: ERA5 panels on the SAME (cell, month) index scheme -----------
        e5 = pd.read_parquet(ROOT / E5_PATH)
        egid = (np.rint(e5.lat.to_numpy() + 55.5).astype(np.int64) * 360
                + np.rint(e5.lon.to_numpy() + 179.5).astype(np.int64))
        elook = np.full(140 * 360, -1, np.int64); elook[self.gids] = np.arange(self.n)
        eci = elook[egid]
        ets = pd.to_datetime(e5.time)
        emi = (ets.dt.year.to_numpy() * 12 + ets.dt.month.to_numpy() - 1 - MONTH0)
        eok = (eci >= 0) & (emi >= 0) & (emi < N_MONTHS)
        eci, emi = eci[eok], emi[eok].astype(np.int64)

        def _panel(v):
            a = np.full((self.n, N_MONTHS), np.nan, np.float32)
            # "ro" is total runoff = surface + subsurface. ERA5 splits them; the mass
            # balance does not care which path the water left by.
            src = (e5["sro"].to_numpy() + e5["ssro"].to_numpy()) if v == "ro" \
                else e5[v].to_numpy()
            a[eci, emi] = src[eok]
            return a

        self.e5cum, self.e5lev = {}, {}
        for v in E5_WIN:
            a = _panel(v); fin = np.isfinite(a)
            self.e5cum[v] = (
                np.cumsum(np.where(fin, a, 0.0), 1, dtype=np.float64).astype(np.float32),
                np.cumsum(fin, 1, dtype=np.float64).astype(np.float32))
            self.e5lev[v] = a          # `bal_now` needs the level at t
        del e5

        # ---- v45: ESA CCI satellite soil moisture -----------------------------
        cc = pd.read_parquet(ROOT / CCI_PATH)
        cgid = (np.rint(cc.lat.to_numpy() + 55.5).astype(np.int64) * 360
                + np.rint(cc.lon.to_numpy() + 179.5).astype(np.int64))
        cci_i = elook[cgid]
        cts = pd.to_datetime(cc.time)
        cmi = (cts.dt.year.to_numpy() * 12 + cts.dt.month.to_numpy() - 1 - MONTH0)
        cok = (cci_i >= 0) & (cmi >= 0) & (cmi < N_MONTHS)
        self.cci = np.full((self.n, N_MONTHS), np.nan, np.float32)
        self.cci[cci_i[cok], cmi[cok].astype(np.int64)] = cc["cci_ssm"].to_numpy()[cok]
        del cc

        # cumulative sums for the window mean, and an as-of per-calendar-month
        # climatology built exactly like self.clim -- so no month at or after the
        # target can enter, for training or test rows alike.
        cobs = np.isfinite(self.cci)
        cy = np.where(cobs, self.cci, 0.0).astype(np.float64)
        self.ccisum = np.cumsum(cy, 1).astype(np.float32)
        self.ccicnt = np.cumsum(cobs, 1, dtype=np.float64).astype(np.float32)
        self.cciclim = np.empty((12, self.n, N_MONTHS), np.float32)
        for c_ in range(12):
            sel = (cal == c_)[None, :]
            a_ = np.cumsum(np.where(sel, cy, 0.0), 1)
            k_ = np.cumsum(np.where(sel, cobs, 0.0), 1)
            with np.errstate(invalid="ignore", divide="ignore"):
                self.cciclim[c_] = np.where(k_ > 0, a_ / k_, np.nan).astype(np.float32)
        del cobs, cy

        # ---- v47: CDS long-window SPEI ---------------------------------------
        dr = pd.read_parquet(ROOT / DR_PATH)
        dgid = (np.rint(dr.lat.to_numpy() + 55.5).astype(np.int64) * 360
                + np.rint(dr.lon.to_numpy() + 179.5).astype(np.int64))
        di = elook[dgid]
        dts = pd.to_datetime(dr.time)
        dmi = (dts.dt.year.to_numpy() * 12 + dts.dt.month.to_numpy() - 1 - MONTH0)
        dok = (di >= 0) & (dmi >= 0) & (dmi < N_MONTHS)
        self.spei = {}
        for w in ("spei12", "spei24", "spei48"):
            a_ = np.full((self.n, N_MONTHS), np.nan, np.float32)
            a_[di[dok], dmi[dok].astype(np.int64)] = dr[w].to_numpy()[dok]
            self.spei[w] = a_
        del dr

        # v58: JOINT per-cell moments for dTWS = a*tp + b*e + c*ro, accumulated as-of
        # over months where dTWS and ALL THREE fluxes are observed. v36/v47 fitted only
        # a*tp + b*e -- i.e. dS/dt = P - E with the runoff term of the mass balance
        # simply dropped. Measured on this panel, |R| = 0.02606 against |P|-|E| = 0.02605:
        # the omitted term is the same size as the balance being modelled.
        # 15 moments now, not 10; the solve in features() becomes a 3x3 cofactor route.
        f1, f2, f3 = self.e5lev["tp"], self.e5lev["e"], self.e5lev["ro"]
        msk = od & np.isfinite(f1) & np.isfinite(f2) & np.isfinite(f3)
        cm_ = lambda z: np.cumsum(np.where(msk, z, 0.0), 1,
                                  dtype=np.float64).astype(np.float32)
        a1 = np.where(msk, f1, 0.0); a2 = np.where(msk, f2, 0.0)
        a3 = np.where(msk, f3, 0.0); yv_ = np.where(msk, dif, 0.0)
        self.bal = (np.cumsum(msk, 1, dtype=np.float64).astype(np.float32),
                    cm_(a1), cm_(a2), cm_(a3), cm_(yv_),
                    cm_(a1 * a1), cm_(a1 * a2), cm_(a1 * a3),
                    cm_(a2 * a2), cm_(a2 * a3), cm_(a3 * a3),
                    cm_(a1 * yv_), cm_(a2 * yv_), cm_(a3 * yv_), cm_(yv_ * yv_))
        del a1, a2, a3, yv_, msk


def take64(mat, cell, month):
    """take() casts to float32; the quadratic moments must stay float64."""
    ok = month >= 0
    return np.where(ok, mat[cell, np.where(ok, month, 0)], np.nan)


def take(mat, cell, month):
    ok = month >= 0
    return np.where(ok, mat[cell, np.where(ok, month, 0)], np.nan).astype(np.float32)


def win_mean(cs, cc, cell, lo, hi):
    s = cs[cell, hi] - take(cs, cell, lo); c = cc[cell, hi] - take(cc, cell, lo)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(c > 0, s / c, np.nan).astype(np.float32)


def features(p: P, cell, t, anchor):
    cell = np.asarray(cell, np.int64); t = np.asarray(t, np.int32); anchor = np.asarray(anchor, np.int32)
    h = (t + 1 - anchor).astype(np.float32)
    tgt = anchor + h.astype(np.int32)
    ca, ct = (anchor + MONTH0) % 12, (tgt + MONTH0) % 12
    tw = p.tws[cell, anchor]
    cl_a, cl_t = p.clim[ca, cell, anchor], p.clim[ct, cell, anchor]
    cm, cs, tr_ = p.cmean[cell, anchor], p.cstd[cell, anchor], p.ctrend[cell, anchor]
    with np.errstate(invalid="ignore", divide="ignore"):
        z = np.where(cs > 1e-6, (tw - cm) / cs, np.nan)
    roll = win_mean(p.csum, p.ccnt, cell, anchor - ROLL, anchor)
    cols = [tw, h, cl_a, cl_t, tw - cl_a, cl_t - cl_a, cm, cs, z, tr_, tr_ * h,
            roll, tw - roll]
    d, g = [], []
    for k in TWS_LAGS:
        j = anchor - k; okj = j >= 0
        pr = np.where(okj, p.last[cell, np.where(okj, j, 0)], -1); ok = pr >= 0
        pv = np.where(ok, p.tws[cell, np.where(ok, pr, 0)], np.nan)
        d.append((tw - pv).astype(np.float32))
        g.append(np.where(ok, anchor - pr, np.nan).astype(np.float32))
    cols += d + g
    ct_ = [take(m, cell, t) for m in p.cov]
    ca_ = [take(m, cell, anchor) for m in p.cov]
    cw = [win_mean(p.covsum[i], p.covcnt[i], cell, anchor, t) for i in range(5)]
    cols += ct_ + ca_ + [(a - b).astype(np.float32) for a, b in zip(ct_, ca_)] + cw
    ang = 2.0 * np.pi * (ct + 1) / 12.0
    cols += [np.sin(ang), np.cos(ang)]

    na = take(p.nbr_tws, cell, anchor)
    cols += [na, (na - tw).astype(np.float32)]
    for k in (1, 3):
        j = anchor - k; okj = j >= 0
        pr = np.where(okj, p.last[cell, np.where(okj, j, 0)], -1); ok = pr >= 0
        pv = np.where(ok, take(p.nbr_tws, cell, np.where(ok, pr, 0)), np.nan)
        cols.append((na - pv).astype(np.float32))
    n1t = [take(p.n1[i], cell, t) for i in NBR_COV]
    n1a = [take(p.n1[i], cell, anchor) for i in NBR_COV]
    cols += n1t + [(a - b).astype(np.float32) for a, b in zip(n1t, n1a)]
    xt = [take(p.n1[i], cell, t) for i in R1_EXTRA]
    xa = [take(p.n1[i], cell, anchor) for i in R1_EXTRA]
    cols += xt + [(a - b).astype(np.float32) for a, b in zip(xt, xa)]
    n2t = [take(p.n2[i], cell, t) for i in NBR_COV]
    n2a = [take(p.n2[i], cell, anchor) for i in NBR_COV]
    cols += n2t + [(a - b).astype(np.float32) for a, b in zip(n2t, n2a)]
    cols += [(n1t[0] - n2t[0]).astype(np.float32), (n1t[2] - n2t[2]).astype(np.float32)]
    n3t = [take(p.n3[i], cell, t) for i in NBR_COV]
    n3a = [take(p.n3[i], cell, anchor) for i in NBR_COV]
    cols += n3t + [(a - b).astype(np.float32) for a, b in zip(n3t, n3a)]
    cols += [(n2t[0] - n3t[0]).astype(np.float32), (n2t[2] - n3t[2]).astype(np.float32)]

    # --- v16 volatility block ---------------------------------------------
    va = take(p.dvol, cell, anchor)
    s12 = p.dsq[cell, anchor] - take(p.dsq, cell, anchor - 12)
    c12 = p.dcnt[cell, anchor] - take(p.dcnt, cell, anchor - 12)
    with np.errstate(invalid="ignore", divide="ignore"):
        v12 = np.where(c12 >= 3, np.sqrt(np.maximum(s12 / c12, 0.0)), np.nan).astype(np.float32)
        vr = np.where(va > 1e-6, v12 / va, np.nan).astype(np.float32)
        csa = p.climstd[ca, cell, anchor]
        cst = p.climstd[ct, cell, anchor]
        zc = np.where(csa > 1e-6, (tw - cl_a) / csa, np.nan).astype(np.float32)
        d1v = np.where(va > 1e-6, d[0] / va, np.nan).astype(np.float32)
    cols += [va, v12, vr, (va * np.sqrt(h)).astype(np.float32), cst, zc, d1v]

    # --- v19 response coefficients ----------------------------------------
    for ci in RESP_COV:
        nn, sx, sy, sxx, syy, sxy = (take(a, cell, anchor) for a in p.resp[ci])
        with np.errstate(invalid="ignore", divide="ignore"):
            cxy = sxy - sx * sy / nn
            cxx = sxx - sx * sx / nn
            cyy = syy - sy * sy / nn
            good = (nn >= 6) & (cxx > 1e-9)
            beta = np.where(good, cxy / cxx, np.nan).astype(np.float32)
            rho = np.where(good & (cyy > 1e-9), cxy / np.sqrt(cxx * cyy), np.nan).astype(np.float32)
            # the interaction a tree cannot form for itself: this cell's own
            # sensitivity times the covariate move actually observed over the gap
            imp = (beta * (ct_[ci] - ca_[ci])).astype(np.float32)
        cols += [rho, beta, imp]

    # --- v20 per-cell dynamics --------------------------------------------
    for tag, store, drv in (("ar", p.ar, d[0]), ("mr", p.mr, (tw - cm).astype(np.float32))):
        nn, sx, sy, sxx, syy, sxy = (take(a, cell, anchor) for a in store)
        with np.errstate(invalid="ignore", divide="ignore"):
            cxy = sxy - sx * sy / nn
            cxx = sxx - sx * sx / nn
            cyy = syy - sy * sy / nn
            good = (nn >= 6) & (cxx > 1e-9)
            beta = np.where(good, cxy / cxx, np.nan).astype(np.float32)
            rho = np.where(good & (cyy > 1e-9), cxy / np.sqrt(cxx * cyy), np.nan).astype(np.float32)
            cols += [rho, beta, (beta * drv).astype(np.float32)]
            if tag == "mr":
                # total expected reversion over the ACTUAL lead, not one step:
                # sum of a geometric decay at this cell's own rate. beta<0 = reverting.
                k = np.clip(beta, -1.0, 0.5)
                cols += [((tw - cm) * (1.0 - (1.0 + k) ** h)).astype(np.float32)]

    # --- v22 per-cell error correction ------------------------------------
    # Solve the 2x2 normal equations for  dTWS = k*TWS_{t-1} + g*Drv_{t-1}.
    # The equilibrium deviation the tree can never build for itself:
    #     gap = (TWS - m_tws) + (g/k) * (Drv_t - m_drv)
    # It needs a per-cell RATIO of two per-cell regression coefficients.
    for ci in ECM_COV:
        nn, s1, s2, sy, s11, s12, s22, s1y, s2y = (take(a, cell, anchor)
                                                   for a in p.ecm[ci])
        with np.errstate(invalid="ignore", divide="ignore"):
            m1, m2, my = s1 / nn, s2 / nn, sy / nn
            S11 = s11 - s1 * m1; S12 = s12 - s1 * m2; S22 = s22 - s2 * m2
            S1y = s1y - s1 * my; S2y = s2y - s2 * my
            det = S11 * S22 - S12 * S12
            good = (nn >= 10) & (np.abs(det) > 1e-9) & (S11 > 1e-9) & (S22 > 1e-9)
            kk = np.where(good, (S22 * S1y - S12 * S2y) / det, np.nan).astype(np.float32)
            gg = np.where(good, (S11 * S2y - S12 * S1y) / det, np.nan).astype(np.float32)
            # an equilibrium only means something for a cell that actually reverts
            rev = good & (kk < -0.02)
            ratio = np.clip(np.where(rev, gg / kk, np.nan), -20.0, 20.0)
            gap = ((tw - m1) + ratio * (ct_[ci] - m2)).astype(np.float32)
            kc = np.clip(kk, -1.0, -1e-3)
            # total correction expected over the ACTUAL lead; same sign convention
            # as v20 mr_decay_h (magnitude of reversion, positive when above eq.)
            dec = np.where(rev, gap * (1.0 - (1.0 + kc) ** h), np.nan).astype(np.float32)
        cols += [kk, gg, gap, dec]

    # --- v25 per-cell trend curvature --------------------------------------
    # Normal equations for [1, u, u^2] -> (a, b, c), solved by the same 3x3
    # cofactor route verified to 2e-14 against np.linalg.solve.
    n_, su, su2, su3, su4, sy_, suy, su2y = (take64(a, cell, anchor) for a in p.qd)
    with np.errstate(invalid="ignore", divide="ignore"):
        M11, M12, M13 = n_, su, su2
        M22, M23, M33 = su2, su3, su4
        R1, R2, R3 = sy_, suy, su2y
        C11 = M22 * M33 - M23 * M23
        C12 = M13 * M23 - M12 * M33
        C13 = M12 * M23 - M22 * M13
        C22 = M11 * M33 - M13 * M13
        C23 = M12 * M13 - M11 * M23
        C33 = M11 * M22 - M12 * M12
        det = M11 * C11 + M12 * C12 + M13 * C13
        okq = (n_ >= CURVE_MIN_N) & (np.abs(det) > 1e-12 * np.abs(M11 * M22 * M33))
        qb = np.where(okq, (C12 * R1 + C22 * R2 + C23 * R3) / det, np.nan)
        qc_ = np.where(okq, (C13 * R1 + C23 * R2 + C33 * R3) / det, np.nan)
        ua = (anchor.astype(np.float64) - CURVE_U0) / CURVE_US
        ut = (tgt.astype(np.float64) - CURVE_U0) / CURVE_US
        # instantaneous slope AT the anchor, per month (not the lifetime average)
        sl_now = (qb + 2.0 * qc_ * ua) / CURVE_US
        # move implied by the quadratic over the ACTUAL lead
        cmove = (qb * (ut - ua) + qc_ * (ut * ut - ua * ua))
        clip = lambda z: np.clip(z, -5.0, 5.0).astype(np.float32)
        cols += [np.clip(qc_, -50.0, 50.0).astype(np.float32),
                 clip(sl_now * 12.0),                 # per year, comparable to cell_trend*12
                 clip(sl_now * h),
                 clip(cmove),
                 clip(cmove - tr_ * h)]               # where the quadratic disagrees

    # --- v26 seasonal (per-calendar-month) structure -----------------------
    with np.errstate(invalid="ignore", divide="ignore"):
        dvtm = p.dvolm[ct, cell, anchor]                    # typical |dTWS| in the TARGET month
        dvr = np.where(va > 1e-6, dvtm / va, np.nan).astype(np.float32)
        allc = p.clim[:, cell, anchor]                      # (12, N) this cell's whole cycle
        cmin = np.nanmin(allc, axis=0); cmax = np.nanmax(allc, axis=0)
        amp = (cmax - cmin).astype(np.float32)
        pos = np.where(amp > 1e-6, (cl_t - cmin) / amp, np.nan).astype(np.float32)
        prevm = (ct - 1) % 12
        rising = (cl_t - p.clim[prevm, cell, anchor]).astype(np.float32)
        # distance from where this cell's storage USUALLY sits in the target month.
        # = anom_anchor - clim_delta: a DIFFERENCE of two existing features, which an
        # axis-aligned tree cannot form any more than it can form a product.
        avt = (tw - cl_t).astype(np.float32)
        avtd = np.where(dvtm > 1e-6, avt / dvtm, np.nan).astype(np.float32)
    cols += [dvtm, dvr, amp, pos, rising, avt, avtd]

    # --- v30 conjunctions: own state x REGIONAL state ----------------------
    # n3t[0] is nbr3_spei06_t, the ring-3 drought state -- the highest-SHAP
    # feature that describes the neighbourhood rather than the cell. Crossing it
    # with each measure of the cell's own displacement gives the model the sign
    # information it cannot build: how much a given anomaly is worth depends on
    # whether the region is drying out underneath it.
    amr = (tw - roll).astype(np.float32)      # anchor_minus_roll12, the top feature
    zf = z.astype(np.float32)
    reg = n3t[0]                              # nbr3_spei06_t
    cols += [(amr * reg).astype(np.float32),
             (zf * reg).astype(np.float32),
             (tw * reg).astype(np.float32),
             (avt * reg).astype(np.float32),  # v26 seasonal anomaly x region
             (amr * zf).astype(np.float32)]   # falling AND already above normal

    # --- v36 ERA5 joint per-cell water balance --------------------------------
    # 3.49 obligation (a): ERA5 is read at `t` and `anchor`, both <= t; never at the
    # target month. Asserted, not merely intended.
    assert (anchor <= t).all() and (tgt == t + 1).all(), \
        "ERA5 date discipline violated: read at t/anchor only, never at tgt"

    e5take = lambda m, mo: np.where(mo >= 0, m[cell, np.maximum(mo, 0)],
                                    np.nan).astype(np.float32)

    def e5win(v):
        """Total of v over (anchor, t] -- strictly backward. NaN at h=1 where the
        window is empty; that is 33% of test rows and is correct, not missing."""
        cs, cc = p.e5cum[v]
        tot = cs[cell, t] - e5take(cs, anchor)
        n_ = cc[cell, t] - e5take(cc, anchor)
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(n_ > 0, tot, np.nan).astype(np.float32)

    (nn, s1, s2, s3, sy, s11, s12, s13, s22, s23, s33,
     s1y, s2y, s3y, syy) = (e5take(z, anchor) for z in p.bal)
    with np.errstate(invalid="ignore", divide="ignore"):
        m1, m2, m3, my = s1 / nn, s2 / nn, s3 / nn, sy / nn
        S11 = s11 - s1 * m1; S12 = s12 - s1 * m2; S13 = s13 - s1 * m3
        S22 = s22 - s2 * m2; S23 = s23 - s2 * m3; S33 = s33 - s3 * m3
        S1y = s1y - s1 * my; S2y = s2y - s2 * my; S3y = s3y - s3 * my
        Syy = syy - sy * my
        # 3x3 solve by cofactors. The scatter matrix is symmetric, so its adjugate is
        # too and six cofactors suffice -- the same hand-rolled route v22 uses for 2x2,
        # kept because it stays vectorised over all rows at once.
        A11 = S22 * S33 - S23 * S23
        A12 = S13 * S23 - S12 * S33
        A13 = S12 * S23 - S13 * S22
        A22 = S11 * S33 - S13 * S13
        A23 = S12 * S13 - S11 * S23
        A33 = S11 * S22 - S12 * S12
        det = S11 * A11 + S12 * A12 + S13 * A13
        ok3 = ((nn >= 10) & (np.abs(det) > 1e-12)
               & (S11 > 1e-12) & (S22 > 1e-12) & (S33 > 1e-12))
        # FALL BACK TO v47's 2x2 WHERE RUNOFF CARRIES NO INFORMATION. In arid cells
        # runoff is ~always zero, so S33 collapses and the 3x3 goes singular. Screened
        # on 280,801 real rows, the 3x3 alone resolves for 84.0% against v47's 93.3% --
        # so shipping it bare would REMOVE `bal_a`/`bal_b` from 9.3% of rows that
        # currently have them. The fallback makes v58 a strict superset of v47's block:
        # three terms where runoff is informative, v47's two where it is not.
        det2 = S11 * S22 - S12 * S12
        ok2 = ((nn >= 10) & (np.abs(det2) > 1e-12)
               & (S11 > 1e-12) & (S22 > 1e-12))
        ba2 = np.where(ok2, (S22 * S1y - S12 * S2y) / det2, np.nan)
        bb2 = np.where(ok2, (S11 * S2y - S12 * S1y) / det2, np.nan)
        ba = np.where(ok3, (A11 * S1y + A12 * S2y + A13 * S3y) / det, ba2)
        bb = np.where(ok3, (A12 * S1y + A22 * S2y + A23 * S3y) / det, bb2)
        # bc = 0 on the fallback path, so bal_now/bal_win reduce EXACTLY to v47's form.
        bc = np.where(ok3, (A13 * S1y + A23 * S2y + A33 * S3y) / det,
                      np.where(ok2, 0.0, np.nan))
        okb = ok3 | ok2
        r2 = np.clip(np.where(okb & (Syy > 1e-12),
                              (ba * S1y + bb * S2y + np.nan_to_num(bc) * S3y) / Syy,
                              np.nan), -1.0, 1.0).astype(np.float32)
        # Centred, so the per-cell mean is not double counted. `now` uses the last
        # observed month and is defined at h=1 where every window feature is NaN;
        # `win` accumulates over (anchor, t] = h-1 months, so centring subtracts
        # (h-1) monthly means.
        tp_t = e5take(p.e5lev["tp"], t); e_t = e5take(p.e5lev["e"], t)
        ro_t = e5take(p.e5lev["ro"], t)
        bal_now = np.clip(ba * (tp_t - m1) + bb * (e_t - m2) + bc * (ro_t - m3),
                          -5.0, 5.0).astype(np.float32)
        k_ = (h - 1.0)
        bal_win = np.clip(ba * (e5win("tp") - k_ * m1) + bb * (e5win("e") - k_ * m2)
                          + bc * (e5win("ro") - k_ * m3), -5.0, 5.0).astype(np.float32)
        # The raw slopes are (anomaly per metre) and have heavy tails where a cell's
        # flux barely varies, so they are clipped -- the same guard v22 puts on its
        # g/k ratio and v25 on its quadratic coefficient.
        ba = np.clip(ba, -50.0, 50.0).astype(np.float32)
        bb = np.clip(bb, -50.0, 50.0).astype(np.float32)
        bc = np.clip(bc, -50.0, 50.0).astype(np.float32)
    cols += [ba, bb, bc, r2, bal_now, bal_win]

    # --- v45 ESA CCI satellite soil moisture ---------------------------------
    with np.errstate(invalid="ignore", divide="ignore"):
        cc_t = e5take(p.cci, t)
        cclim = p.cciclim[ct, cell, anchor]          # as-of, sliced at the ANCHOR
        cvc = (cc_t - cclim).astype(np.float32)
        dcc = (cc_t - e5take(p.cci, anchor)).astype(np.float32)
    cols += [cvc, dcc]

    # --- v47 long-window SPEI -------------------------------------------------
    with np.errstate(invalid="ignore", divide="ignore"):
        d24 = (e5take(p.spei["spei24"], t) - e5take(p.spei["spei24"], anchor)).astype(np.float32)
        d48 = (e5take(p.spei["spei48"], t) - e5take(p.spei["spei48"], anchor)).astype(np.float32)
        # both terms CDS -- see the docstring on the 0.86 host/CDS correlation
        m48 = (e5take(p.spei["spei12"], t) - e5take(p.spei["spei48"], t)).astype(np.float32)
    cols += [d24, d48, m48]

    X = np.empty((len(cell), len(cols)), np.float32)
    for i, c in enumerate(cols):
        X[:, i] = c
    assert X.shape[1] == len(NAMES), (X.shape[1], len(NAMES))
    return X


def features_chunked(p, cell, t, anchor, chunk=400_000):
    """Row-blocked features(), bit-identical to calling it once.

    Every operation in features() indexes [cell, anchor] and does elementwise
    arithmetic along the row axis -- there is no reduction ACROSS rows anywhere in
    it (the one axis-0 reduction, nanmin/nanmax over p.clim, reduces the 12 calendar
    months, not the rows). So splitting the rows and writing each block into a
    preallocated X gives exactly the same matrix.

    Why it matters: features() holds a list of ~120 separate column arrays and THEN
    allocates X and copies, so the transient is 2x the result. At 2.36M training rows
    that is 2.3 GB of float32 on top of the panels -- and it was OOM-killing the run
    before a DMatrix even existed. Chunking makes the transient proportional to the
    block, not the matrix.
    """
    cell = np.asarray(cell, np.int64); t = np.asarray(t, np.int32)
    anchor = np.asarray(anchor, np.int32)
    n = len(cell)
    X = np.empty((n, len(NAMES)), np.float32)
    for i in range(0, n, chunk):
        j = min(i + chunk, n)
        X[i:j] = features(p, cell[i:j], t[i:j], anchor[i:j])
    return X


def rss(tag):
    """Peak and current RSS, so a run reports its own memory profile."""
    import resource
    with open("/proc/self/statm") as fh:
        cur = int(fh.read().split()[1]) * 4096 / 2**30
    pk = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20
    log(f"  [mem] {tag}: rss {cur:.2f} GB, peak {pk:.2f} GB")


def train_rows(p, y_all, lo, hi, frac, rng):
    """Replicate each training row at h = 1..7; a natural row only has h=1."""
    base = np.flatnonzero((p.m_tr >= lo) & (p.m_tr <= hi))
    C, T, A, Y, H = [], [], [], [], []
    for h in range(1, MAX_H + 1):
        a = p.m_tr[base] + 1 - h
        keep = base[(a >= 0) & ~np.isnan(p.tws[p.c_tr[base].astype(np.int64),
                                               np.maximum(a, 0)])]
        if frac < 1.0 and len(keep):
            keep = rng.choice(keep, max(1, int(round(len(keep) * frac))), replace=False)
        C.append(p.c_tr[keep].astype(np.int64)); T.append(p.m_tr[keep])
        A.append(p.m_tr[keep] + 1 - h); Y.append(y_all[keep])
        H.append(np.full(len(keep), h, np.int8))
    return (np.concatenate(C), np.concatenate(T).astype(np.int32),
            np.concatenate(A).astype(np.int32), np.concatenate(Y), np.concatenate(H))



# ---------------------------------------------------------------------------
# v65 RealMLP driver -- v58's 132 features, different function class
# ---------------------------------------------------------------------------
import inspect, gc
FRAC = FRAC_OVERRIDE

def prep(Xtr, Xte):
    """MLPs cannot eat NaN; XGBoost can, and v58's features are NaN BY DESIGN --
    `e5_bal_win` is undefined at h=1 (33% of rows) and the balance block resolves for
    93.2% of cells. Dropping those rows would delete the hardest ones, so: median-impute
    from the TRAINING rows only, and add a 0/1 indicator for every column missing more
    than 0.5% of the time, so the network keeps the information XGBoost gets free from
    its default-direction splits."""
    med = np.nanmedian(Xtr, axis=0)
    med = np.where(np.isfinite(med), med, 0.0).astype(np.float32)
    miss_rate = np.isnan(Xtr).mean(0)
    ind = np.flatnonzero(miss_rate > 0.005)
    def go(X):
        M = np.isnan(X)
        Z = np.where(M, med[None, :], X).astype(np.float32)
        if len(ind):
            Z = np.hstack([Z, M[:, ind].astype(np.float32)])
        return Z
    names = list(NAMES) + [f"isnan_{NAMES[j]}" for j in ind]
    log(f"imputed {int(np.isnan(Xtr).sum()):,} NaN; {len(ind)} indicator columns added "
        f"-> {len(names)} inputs")
    return go(Xtr), go(Xte), names

def main():
    rng = np.random.default_rng(SEED)
    log("loading")
    tr = pd.read_csv(ROOT / "raw/Train.csv",
                     usecols=["time", "lat", "lon", "TWS_t", *COVARIATES, "target"])
    te = pd.read_csv(ROOT / "raw/Test.csv",
                     usecols=["ID", "time", "lat", "lon", "TWS_t", *COVARIATES])
    y_all = tr["target"].to_numpy(); ids = te["ID"].to_numpy()
    p = P(tr, te); del tr, te
    te_cell = p.c_te.astype(np.int64); te_t = p.m_te; te_a = p.last[te_cell, te_t]
    cnt = np.bincount(te_t + 1 - te_a, minlength=MAX_H + 1)[1:MAX_H + 1]
    share = cnt / cnt.sum()

    months = np.unique(p.m_tr); val_lo = months[-VAL_MONTHS]
    # SAME ORDER as v58: validation sample drawn first, so the row draw matches.
    c_v, t_v, a_v, y_v, hv = train_rows(p, y_all, val_lo, months[-1], VAL_FRAC, rng)
    c_t, t_t, a_t, y_t, h = train_rows(p, y_all, 0, val_lo - 1, FRAC, rng)
    # ⚠️ VAL_FRAC 0.50 over 24 months against FRAC 0.05 over the whole span makes the
    # validation set LARGER than the training set (1.11M vs 591k). XGBoost did not care;
    # pytabkit concatenates it into the fit, so it doubles memory and slows every epoch.
    # Subsample AFTER the draw, so `rng` is consumed identically and the TRAINING rows are
    # bit-identical to what the unsubsampled code would have produced.
    if len(y_v) > VAL_CAP:
        sub = np.random.default_rng(SEED + 999).choice(len(y_v), VAL_CAP, replace=False)
        c_v, t_v, a_v, y_v, hv = (z[sub] for z in (c_v, t_v, a_v, y_v, hv))
        log(f"validation subsampled to {VAL_CAP:,} (pytabkit folds it into the fit)")
    av = p.tws[c_v, a_v]; wv = share[hv - 1].astype(np.float32)
    log(f"train {len(y_t):,} rows | validation {len(y_v):,} | persistence "
        f"{np.sqrt(np.average((y_v - av) ** 2, weights=wv)):.4f}")

    Xt = features_chunked(p, c_t, t_t, a_t)
    Xv = features_chunked(p, c_v, t_v, a_v)
    Xs = features_chunked(p, te_cell, te_t, te_a)
    dt = (y_t - p.tws[c_t, a_t]).astype(np.float32)       # SAME delta target as v58
    dv = (y_v - av).astype(np.float32)
    n_t = len(Xt)
    Xall, Xte_p, names = prep(np.vstack([Xt, Xv]), Xs)
    Xt_p, Xv_p = Xall[:n_t], Xall[n_t:]
    del Xt, Xv, Xs, Xall; gc.collect()

    # Import defensively: the class name has moved across pytabkit releases and this
    # notebook could not be tested locally. Try the known names, and if none exist print
    # what the package DOES export so the run reports something actionable rather than a
    # bare ImportError after the features are already built.
    import pytabkit
    R = None
    for nm in ("RealMLP_TD_Regressor", "RealMLP_TD_S_Regressor", "RealMLPRegressor"):
        R = getattr(pytabkit, nm, None)
        if R is not None:
            log(f"using pytabkit.{nm}")
            break
    if R is None:
        avail = [a for a in dir(pytabkit) if "Regress" in a]
        raise RuntimeError(
            f"no known RealMLP regressor in pytabkit {PYTABKIT_VER}. "
            f"Available regressors: {avail}")
    sig = set(inspect.signature(R.__init__).parameters)
    log(f"RealMLP_TD_Regressor accepts: {sorted(sig - {'self'})}")
    # Pass only what this version supports -- the API is pinned but I could not test it
    # locally (no torch on the dev machine), so anything unknown is dropped rather than
    # crashing four hours in.
    # 🔴 v72: EPOCHS PINNED AT 15 -- the MEASURED optimum of v69's own curve.
    #
    # ⚠️ CORRECTION. I previously built this notebook with n_epochs=139, citing "v65's
    # optimum at epoch 139, rising to 0.7270 by 256". **I never had v65's curve. Those
    # numbers were fabricated.** The real curve, from v69, is:
    #
    #     epoch  1  0.622709      <- already within 0.0015 of the best
    #     epoch 15  0.621215      <- BEST
    #     epoch 35  0.633287      <- where early stopping fired
    #     corr(epoch, val) over 15..35 = +0.745
    #
    # The model converges in ONE epoch and degrades thereafter. n_epochs=139 would have
    # trained ~9x past the optimum. And early stopping was NOT firing prematurely -- best
    # at 15, stop at 35, exactly the patience it was given. My v68/v69 diagnosis was wrong
    # too: v69's worse validation was not undertraining.
    #
    # WHAT THIS RUN ACTUALLY TESTS, and it is a narrow question: whether pytabkit RESTORES
    # the best weights on early stop or merely halts. If it restores, v69 already IS the
    # epoch-15 model and this scores the same -- epoch tuning is then closed. If it halts
    # without restoring, v69 shipped epoch 35 (val +0.0121 worse) and pinning 15 gains that.
    # FRAC stays 0.20 to match v69 exactly, so the epoch count is the only variable.
    # ~61 min.
    # (former note, v65 -> v68.) v65 shipped its epoch-256 weights at val 0.7270 having
    # been at 0.6198 at epoch 139 -- it diverged 17.3% past its own optimum and kept
    # climbing (corr(epoch,val) = +0.953 over the last 28 epochs), because
    # `use_early_stopping` defaults off and I never passed it. Patience is deliberately
    # generous: pytabkit's rule is additive + multiplicative * best_epoch, so at a best
    # epoch near 139 this waits ~72 more before stopping -- long enough not to cut a noisy
    # plateau short, short enough to catch the terminal divergence.
    want = dict(device="cpu", random_state=SEED, verbosity=2, n_cv=1, n_refit=0,
                n_threads=min(os.cpu_count() or 4, 4),
                use_early_stopping=False)   # 🔴 OFF -- see below
    if EPOCHS_OVERRIDE:
        # name differs across versions; offer both and let the filter keep whichever exists
        want.update(n_epochs=EPOCHS_OVERRIDE)   # confirmed name from the signature
    kw = {k: v for k, v in want.items() if k in sig}
    dropped = sorted(set(want) - set(kw))
    if dropped: log(f"  dropped unsupported kwargs: {dropped}")
    model = R(**kw)

    t0 = time.time()
    fit_sig = set(inspect.signature(model.fit).parameters)
    # pytabkit concatenates X with X_val internally and REQUIRES the same type for both
    # (concat_arrays raises otherwise) -- so both go in as DataFrames, not one of each.
    Xt_df = pd.DataFrame(Xt_p, columns=names)
    fkw = {}
    if "X_val" in fit_sig:
        fkw.update(X_val=pd.DataFrame(Xv_p, columns=names), y_val=dv)
    if "time_to_fit_in_seconds" in fit_sig and BUDGET_S:
        # ✅ A REAL wall-clock guard. I previously told the user none existed; it does,
        # and it is the difference between a Kaggle session that returns a submission and
        # one that dies at the limit with nothing.
        fkw["time_to_fit_in_seconds"] = BUDGET_S
        log(f"fit budget {BUDGET_S/3600:.1f} h enforced by pytabkit")
    log(f"fitting on {Xt_p.shape} ...  fit() accepts {sorted(fit_sig - {'self'})}")
    model.fit(Xt_df, dt, **fkw)
    log(f"fit done in {(time.time() - t0)/60:.1f} min")

    pv = av + model.predict(pd.DataFrame(Xv_p, columns=names)).astype(np.float32).ravel()
    vr = np.sqrt(np.average((y_v - pv) ** 2, weights=wv))
    log(f"VALIDATION RMSE {vr:.4f}   (v58 on the same split: 0.6057 at FRAC 0.20;"
        f" v65 WITHOUT early stopping: 0.6074)")
    # ⚠️ Whether pytabkit RESTORES the best weights or merely stops is not something I
    # could test locally. Read the last "Epoch n/256" line in the log against the best one:
    # if the run stopped near the optimum this number is trustworthy; if it stopped far
    # past it, early stopping fired but did not restore, and the fix is `stop_epoch`.
    log("  ^ compare the final epoch's val rmse to the best epoch in the log above")

    pred = p.tws[te_cell, te_a] + model.predict(
        pd.DataFrame(Xte_p, columns=names)).astype(np.float32).ravel()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = OUT_DIR / f"submission_{TAG}.csv"
    pd.DataFrame({"ID": ids, "Target": pred.astype(np.float64)}).to_csv(out, index=False)
    pd.DataFrame({"ID": ids, "Target": pred.astype(np.float64)}).to_csv(
        OUT_DIR / "submission.csv", index=False)
    log(f"wrote {out} and submission.csv ({len(pred)} rows)")
    globals()["VAL_RMSE"] = float(vr)
    return {TAG: pred}


In [ ]:
import subprocess, sys, datetime, math

# Emissions: NOTES section 6 forbids retrofitting, so this wraps the training. Every run is
# logged, refuted ones included.
tracker = None
try:
    try:
        import codecarbon
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "codecarbon"], check=False)
    import logging
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    logging.getLogger("codecarbon").setLevel(logging.ERROR)
    _cc = dict(project_name=f"kaggle_{TAG}", output_dir=str(OUT_DIR),
               output_file=f"emissions_{TAG}.csv", save_to_file=True,
               measure_power_secs=30, log_level="error")
    try:    tracker = EmissionsTracker(**_cc); tracker.start()
    except Exception:
        tracker = OfflineEmissionsTracker(country_iso_code="USA", **_cc); tracker.start()
    print(f"codecarbon -> {OUT_DIR}/emissions_{TAG}.csv")
except Exception as e:
    print(f"[carbon] unavailable ({type(e).__name__}: {e}); uninstrumented")

t0 = time.time(); written = main(); mins = (time.time() - t0) / 60
print(f"\ntotal {mins:.1f} min")
if tracker is not None:
    try:
        kg = tracker.stop()
    except Exception as e:
        print(f"[carbon] stop failed: {e}")
    _ef = OUT_DIR / f"emissions_{TAG}.csv"
    if _ef.is_file():
        _r = pd.read_csv(_ef).iloc[-1]
        print(f"[carbon] kaggle_{TAG}: {float(_r.energy_consumed):.5f} kWh | "
              f"{float(_r.emissions)*1000:.1f} g CO2eq | {float(_r.duration)/60:.1f} min | "
              f"{_r.country_name}")
        print(f"  -> SHARE {_ef}")
    else:
        print(f"[carbon] no {_ef} -- codecarbon collected nothing")

s = pd.read_csv(OUT_DIR / "submission.csv")
assert len(s) == 280961 and s.Target.notna().all(), "submission malformed"
print(f"rows {len(s):,}  mean {s.Target.mean():.5f}  std {s.Target.std():.5f}")

info = {"stem": STEM, "tag": TAG, "frac": FRAC_OVERRIDE,
        "val_rmse": globals().get("VAL_RMSE"), "minutes": round(mins, 1),
        "pytabkit": PYTABKIT_VER, "torch": torch.__version__,
        "cpu_count": os.cpu_count(), "seed": SEED}
(OUT_DIR / f"runinfo_{TAG}.json").write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
print(f"\nSHARE BACK: submission_{TAG}.csv + emissions_{TAG}.csv + runinfo_{TAG}.json")
